# 09 Cross Encoder Reranking

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `09-cross-encoder-reranking.ipynb`

In [1]:
# Start coding here
# ==================================================
# Notebook 09
# Cross Encoder Reranking
# ==================================================

import json
import pickle
import faiss
import pandas as pd
import numpy as np

from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, CrossEncoder

from sklearn.metrics.pairwise import cosine_similarity

d:\Books\projects-nlp-transformers-learning\.projectnlps\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [3]:
children_df = pd.read_csv("artifacts/child_chunks.csv")

parents_df = pd.read_csv("artifacts/parent_chunks.csv")

metadata_df = pd.read_csv("artifacts/document_metadata.csv")

In [4]:
with open("artifacts/router_embeddings.pkl", "rb") as file:

    department_embeddings = pickle.load(file)

In [5]:
vector_indexes = {}

for dept in ["hr", "finance", "engineering"]:

    vector_indexes[dept] = faiss.read_index(f"artifacts/faiss/{dept}.index")

In [6]:
with open("artifacts/faiss/chunk_mapping.pkl", "rb") as file:

    chunk_mappings = pickle.load(file)

In [7]:
children_df = children_df.merge(
    metadata_df[["document_id", "department"]], on="document_id", how="left"
)

department_bm25 = {}
department_docs = {}

for dept in children_df["department"].unique():

    subset = children_df[children_df["department"] == dept].reset_index(drop=True)

    corpus = [text.lower().split() for text in subset["content"]]

    department_bm25[dept] = BM25Okapi(corpus)

    department_docs[dept] = subset

In [8]:
def route_query(query):

    query_embedding = embedding_model.encode(query)

    scores = {}

    for dept, dept_embedding in department_embeddings.items():

        score = cosine_similarity(
            query_embedding.reshape(1, -1), dept_embedding.reshape(1, -1)
        )[0][0]

        scores[dept] = float(score)

    best = max(scores, key=scores.get)

    return best

In [9]:
def get_parent_context(parent_id):

    result = parents_df[parents_df["parent_id"] == parent_id]

    if len(result) == 0:
        return ""

    return result.iloc[0]["content"]

In [10]:
def bm25_search(query, department, top_k=20):

    bm25 = department_bm25[department]

    docs = department_docs[department].copy()

    scores = bm25.get_scores(query.lower().split())

    docs["score"] = scores

    docs = docs.sort_values("score", ascending=False)

    return docs.head(top_k)

In [11]:
def semantic_search(query, department, top_k=20):

    query_embedding = embedding_model.encode([query]).astype("float32")

    faiss.normalize_L2(query_embedding)

    index = vector_indexes[department]

    scores, ids = index.search(query_embedding, top_k)

    mapping = chunk_mappings[department]

    results = []

    for score, idx in zip(scores[0], ids[0]):

        row = mapping.iloc[idx]

        results.append(
            {
                "parent_id": row["parent_id"],
                "content": row["content"],
                "score": float(score),
            }
        )

    return results

In [12]:
def hybrid_candidates(query, top_k=20):

    department = route_query(query)

    bm25_results = bm25_search(query, department, top_k)

    semantic_results = semantic_search(query, department, top_k)

    candidates = {}

    for _, row in bm25_results.iterrows():

        candidates[row["parent_id"]] = get_parent_context(row["parent_id"])

    for row in semantic_results:

        candidates[row["parent_id"]] = get_parent_context(row["parent_id"])

    return list(candidates.items())

In [13]:
def rerank_documents(query, documents, top_k=5):

    pairs = []

    for _, text in documents:

        pairs.append([query, text])

    scores = cross_encoder.predict(pairs)

    ranked = []

    for (doc_id, text), score in zip(documents, scores):

        ranked.append({"parent_id": doc_id, "score": float(score), "content": text})

    ranked = sorted(ranked, key=lambda x: x["score"], reverse=True)

    return ranked[:top_k]

In [14]:
def retrieve_and_rerank(query):

    candidates = hybrid_candidates(query)

    ranked = rerank_documents(query, candidates)

    return ranked

In [15]:
results = retrieve_and_rerank("What AI infrastructure projects are planned?")

for r in results:

    print("=" * 80)

    print("Score:", round(r["score"], 4))

    print(r["content"])

Score: -2.1703
AI Platform Roadmap
Search Infrastructure
Retrieval Systems
Deployment Roadmap
Vector search infrastructure rollout.
Search latency improved significantly.
New indexing pipeline deployed.
Hybrid retrieval implementation.
Semantic search quality improved.
Knowledge base coverage expanded.
Cross encod
Score: -6.8881
er deployment planned.
Enterprise RAG platform launch Q3.
Agentic workflows under evaluation.
Component | Status
Vector Search | Production
Hybrid Retrieval | Testing
Cross Encoder | Planned

